In [22]:
!pip install langchain_huggingface langchain_ollama langchain_google_genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.7/70.7 kB 833.2 kB/s eta 0:00:00


In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_huggingface import HuggingFacePipeline
from langchain_core.tools import tool
import requests
from langchain_core.messages import HumanMessage

In [27]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    google_api_key="AQ.Ab8RN6LnsnWvN5VcM4j_oWKynkEQrAChX4gemjDxLwq_zhIe8Q",
)

In [30]:
# from langchain_ollama import ChatOllama

# llm = ChatOllama(model="qwen3:8b")

In [31]:
# Tool create

@tool
def multiply(a: int, b:int) -> int:
  """Multiply two numbers and return integer output"""
  return a*b

In [32]:
print(multiply.invoke({
    "a":3,
    "b": 3
}))

9


In [33]:
# Tool Binding

In [28]:
llm_with_tools = llm.bind_tools([multiply])

In [29]:
llm_with_tools.invoke('Hi How are you')

AIMessage(content=[{'type': 'text', 'text': "Hello! I'm doing well, thank you for asking. How can I help you today?", 'extras': {'signature': 'EvQDCvEDARFNMg+WGLEp7xhYIF47P6wcCD8e7nIDhqlk84jFbGPp623j5PB8ogWT/5YJAefh+tcOhCqBr7bzfRXmRLJ98iIxlo9Tk1Q9AlNoUoarYfX6ko1nqP6NREkIfu9F4kQ9Wp0e+pIcq3DJolMh6i22xjSJkh6yWbPSJiQZ5bTGriBQaA0dfC7gAwX0sJwyroY1t+gmexzPeTzXAoZOpcJV144zOTQ+flvsWxAoSV/PHNsjPyEmk1fpUtslDexO9Tb4dRhNqkWP51sVkbLyOkhr+mYF9TfiiHMeNvmopM/T1JPUfU3pRkXru9lAFcmuqu4yGS+FaglZG5W6O/HVt3SNeCSMcmvV+7XGB7/hZqeBtCVvVT8mpILwz+DG5gAUHw6cAQmL58wlvGDPhmGSgePabacybtfCzVIj7VsKurPAQn6UwFeI7UYE+eUjdX0QHseEjazH9sJLX9Eqlr2v6H+cyZ/wEemlB7+mHFJEodNKlQdhZCHgbuzf1Z5eajvWZEbFYVlZpfNk4gi2vwjG2QoPNdte9fKjaLIqyGEnnuliAJrNnTITukUZn6+XUU7AvwqnxuPw9grzocc4Q045DWuC0MRMxjCzovMqUV1Og5YL1Hnhwn1gIduB4865i2qTv0PfTBpeEcLV0jsPy82bgbpOxX0='}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f54a6-a

In [48]:
query = HumanMessage('Can you multiply 3 with 10')

In [49]:
messages = [query]

In [50]:
messages

[HumanMessage(content='Can you multiply 3 with 10', additional_kwargs={}, response_metadata={})]

In [51]:
result = llm_with_tools.invoke(messages)

In [52]:
messages.append(result)

In [53]:
result.tool_calls[0]['args']

{'a': 3, 'b': 10}

In [54]:
tool_result = multiply.invoke(result.tool_calls[0])

In [55]:
messages.append(tool_result)

In [56]:
messages

[HumanMessage(content='Can you multiply 3 with 10', additional_kwargs={}, response_metadata={}),
 AIMessage(content=[], additional_kwargs={'function_call': {'name': 'multiply', 'arguments': '{"a": 3, "b": 10}'}, '__gemini_function_call_thought_signatures__': {'g5xul4sy': 'EvUBCvIBARFNMg8Zw0z1vUqy9MCvZSsIBfO9b6z8w9gm5nlIUrssdPdVpD5vgLKd8jrm7kllkbOoyhAhnYfBkfqNgXwFchp7L57C4Y1cvmk8iaXwLG34pWqLimRAX2SUDKLsXOjFDD14EXPnO/NNxn34+kCV4Dr8+20WsNbtnCwTXCWiYe7732UNRR5G+8r0iKKgoSbEBPOLgHfnCHN3yMkGFzFvqtM7TKkBsDJvP+SItoZgHZ8f97YbB/2SzTwan+zgiGlowcD0V+xwDP+7/tYxXuVkqXSMBHBkqA5z1x8HiQaJ9yrUPD4BNZmoyAEttvC/nitNL5s='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f54b4-2325-7501-86a9-7cb4e5beb58c-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 10}, 'id': 'g5xul4sy', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 64, 'output_tokens': 76, 'total_tokens': 140, 

In [66]:
llm_with_tools.invoke(messages).content[0].get("text")

'3 * 10 = 30'